In [ ]:
import os
DOSSIER = r"C:\Users\stgadmin\Desktop\TFE-STIB"
print(os.listdir(DOSSIER))

In [ ]:
import zipfile, csv, pyodbc, os

DOSSIER = r"C:\Users\stgadmin\Desktop\TFE-STIB"   

ZIPS = [
    "STIB_22.06_19.07.zip",
    "STIB_06.07_02.08.zip",
    "STIB_03.08_30.08.zip",
]

FICHIERS = {
    "stops.txt":          "stg_stops",
    "routes.txt":         "stg_routes",
    "trips.txt":          "stg_trips",
    "stop_times.txt":     "stg_stop_times",
    "calendar.txt":       "stg_calendar",
    "calendar_dates.txt": "stg_calendar_dates",
}

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ICT-202-11;DATABASE=TFE_STIB;Trusted_Connection=yes;"
)
conn.autocommit = False
cur = conn.cursor()
cur.fast_executemany = True


def colonnes_de_la_table(cursor, table):
    """Retourne l'ensemble des colonnes existantes dans la table SQL."""
    cursor.execute("""
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = ?
    """, table)
    return {r[0].lower() for r in cursor.fetchall()}


for nom_zip in ZIPS:
    chemin = os.path.join(DOSSIER, nom_zip)
    if not os.path.exists(chemin):
        print(f"[SKIP] {nom_zip} introuvable")
        continue

    with zipfile.ZipFile(chemin) as z:
        noms = z.namelist()

        # --- version du feed, lue dans le fichier lui-même ---
        version = os.path.splitext(nom_zip)[0]
        if "feed_info.txt" in noms:
            with z.open("feed_info.txt") as f:
                lignes = list(csv.reader(l.decode("utf-8-sig") for l in f))
                if len(lignes) >= 2 and "feed_version" in lignes[0]:
                    version = lignes[1][lignes[0].index("feed_version")]

        print(f"\n=== {nom_zip}  ->  feed_version = {version} ===")

        for fichier, table in FICHIERS.items():
            if fichier not in noms:
                print(f"  [SKIP] {fichier} absent")
                continue

            cols_sql = colonnes_de_la_table(cur, table)

            with z.open(fichier) as f:
                texte = (l.decode("utf-8-sig") for l in f)
                lecteur = csv.reader(texte)
                entetes = [h.strip().lower() for h in next(lecteur)]

                # on ne garde que les colonnes du CSV présentes dans la table
                garder = [i for i, h in enumerate(entetes) if h in cols_sql]
                noms_cols = [entetes[i] for i in garder]

                ignorees = [h for h in entetes if h not in cols_sql]
                if ignorees:
                    print(f"  [INFO] colonnes ignorées dans {fichier} : {ignorees}")

                liste_cols = ", ".join(["feed_version"] + noms_cols)
                placeholders = ", ".join(["?"] * (len(noms_cols) + 1))
                sql = f"INSERT INTO {table} ({liste_cols}) VALUES ({placeholders})"

                lot, total = [], 0
                for ligne in lecteur:
                    if not ligne:
                        continue
                    ligne = ligne + [None] * (len(entetes) - len(ligne))
                    lot.append([version] + [ligne[i] for i in garder])

                    if len(lot) >= 10000:
                        cur.executemany(sql, lot)
                        total += len(lot)
                        lot = []

                if lot:
                    cur.executemany(sql, lot)
                    total += len(lot)

            conn.commit()
            print(f"  [OK] {fichier} -> {table} : {total:,} lignes")

cur.close()
conn.close()
print("\nImport terminé.")